# Introduction to Improving Telescope Images with AI 🚀🔭
*By: Gabriel Missael Barco, Nicolas Payot, Auriane Thilloy, Olivia Pereira*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/GabrielMissael/super-resolution-workshop/blob/master/notebooks/Diffusion_Simulated_Galaxy_Pipeline.ipynb
)      [![View on GitHub](https://img.shields.io/badge/View_on-GitHub-black?logo=github)](
https://github.com/GabrielMissael/super-resolution-workshop
)

Welcome! In this notebook we’ll see how **astronomy** and **machine learning** work together.  
Real telescope images can be blurry, noisy and low-resolution. We’ll learn how to:

1. Simulate how a telescope distorts a clean galaxy image.
2. Use a powerful AI model (a **diffusion model**) that has learned what galaxies usually look like.
3. Combine both pieces to recover a sharp, high-resolution view of a galaxy from a noisy observation.
4. Finish with a **mystery galaxy challenge** 👀

You only need basic Python; we’ll keep the math light and focus on **ideas + visuals**.


## 0. Hidden helper code 🧰

This cell defines all the **behind-the-scenes tools** we’ll use later:

- Functions to plot images nicely,
- Code that simulates telescope effects (blur, downsampling, noise),
- And a small inference engine that talks to our diffusion model.

You don’t need to read or understand this whole cell to follow the notebook.  
Think of it as the **engine under the hood** — we’ll drive the car in the next cells 🚗✨.


In [ ]:
!git clone --quiet https://github.com/GabrielMissael/super-resolution-workshop
!pip3 install -q git+https://github.com/AlexandreAdam/score_models.git@dev

In [ ]:
import sys
sys.path.append("super-resolution-workshop")

from src.diffusion_sampling.diffusion_sampling import *

## 1. Loading our galaxy data and AI model 🌌🤖

Now we load two important things:

- A small dataset of **clean galaxy images** (what the sky really looks like in our toy universe).
- A pre-trained **diffusion model** that has learned the “language of galaxies” from many examples.

At the end of the cell we quickly visualize a few galaxies to get a feeling for the data we’ll be working with.


In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download
from score_models import ScoreModel
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Using device:", DEVICE)

# Where we want the model to live in the notebook filesystem
MODEL_DIR = Path("model/galaxy_prior")

if not MODEL_DIR.exists():
    print("Downloading galaxy prior from Hugging Face…")
    snapshot_download(
        repo_id="GMissaelBarco/galaxy-prior",
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,  # safer on Colab
    )

# Load pre-trained diffusion model (galaxy prior)
model = ScoreModel(path=str(MODEL_DIR)).to(DEVICE)
model.load()
model.eval()
print("Model loaded ✓")


In [ ]:
# Load galaxy dataset
galaxies = torch.load("super-resolution-workshop/data/galaxies.pt", map_location=DEVICE)
print("Galaxies shape:", galaxies.shape)

# Quick look at the first 5 galaxies
show_grid(galaxies, title="Example clean galaxies")


## 2. Telescope effect #1: blur (the PSF) 👓

Real telescopes never produce perfectly sharp images.  
Because of optics and the atmosphere, light from a single point spreads out — this is described by the **Point Spread Function (PSF)**.

In this interactive cell you can:

- Change the PSF width (`Sigma PSF`),
- See how the galaxies become more or less blurry.

Play with the slider and watch how increasing the PSF smears the details of the spiral arms.


In [ ]:
sigma_psf_slider_explore = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=5.0,
    step=0.01,
    description="Sigma PSF:",
    layout=widgets.Layout(width="800px"),
)

out_psf = widgets.Output()

def update_psf_explore(change=None):
    with out_psf:
        clear_output(wait=True)
        psf_images = psf_on_image(galaxies, sigma=float(sigma_psf_slider_explore.value))
        show_grid(psf_images, title="Effect of PSF (blur only)")

sigma_psf_slider_explore.observe(update_psf_explore, names="value")
display(sigma_psf_slider_explore)
update_psf_explore()
display(out_psf)


## 3. Telescope effect #2: limited resolution 🔍➡️🧱

Detectors are made of **pixels**, and we only have a finite number of them.  
If we use fewer pixels, we lose fine details even if there is no blur.

Here you can:

- Change the number of pixels we keep (`Pixels downsampled to`),
- See how the same galaxy looks at different resolutions.

Notice how small structures disappear as the resolution goes down.


In [ ]:
pixels_downsample_slider_explore = widgets.IntSlider(
    value=64,
    min=8,
    max=64,
    step=1,
    description="Pixels downsampled to:",
    layout=widgets.Layout(width="800px"),
)

out_downsample = widgets.Output()

def update_downsample_explore(change=None):
    with out_downsample:
        clear_output(wait=True)
        downsampled = downsample_img(galaxies, size=int(pixels_downsample_slider_explore.value))
        show_grid(downsampled, title="Effect of downsampling (resolution only)")

pixels_downsample_slider_explore.observe(update_downsample_explore, names="value")
display(pixels_downsample_slider_explore)
update_downsample_explore()
display(out_downsample)


## 4. Telescope effect #3: random noise 🌧️

Even with a perfect telescope, our images are affected by **noise**:

- Photons arrive randomly,
- The detector electronics add extra randomness.

This cell lets you:

- Increase or decrease the **noise level**,
- See how the image becomes grainy and harder to interpret.

Try strong noise and imagine how hard it would be to measure galaxy shapes from such data!


In [ ]:
sigma_noise_slider_explore = widgets.FloatSlider(
    value=0.00,
    min=0.0,
    max=0.5,
    step=0.0005,
    description="Noise σ:",
    layout=widgets.Layout(width="800px"),
)

out_noise = widgets.Output()

def update_noise_explore(change=None):
    with out_noise:
        clear_output(wait=True)
        noisy = add_gaussian_noise(galaxies, sigma=float(sigma_noise_slider_explore.value))
        show_grid(noisy, title="Effect of Gaussian noise")

sigma_noise_slider_explore.observe(update_noise_explore, names="value")
display(sigma_noise_slider_explore)
update_noise_explore()
display(out_noise)


## 5. Putting telescope effects together 🧪

In reality, all these effects happen **at the same time**:

1. The galaxy is blurred by the PSF,
2. Its light is collected on a finite pixel grid (downsampling),
3. Noise is added on top.

The sliders below control these three steps. The panel shows:

- Five galaxies after **blur + downsampling + noise**,
- A green frame highlights the galaxy we’ll later try to recover.

Play with the sliders to build a realistic (or extreme!) telescope configuration, then pick your favourite galaxy for the reconstruction task.

Which parameter (PSF, resolution, noise) hurts the image most in your opinion? Why?

In [ ]:
# Shared sliders for the forward model used later in inference
sigma_psf_slider = widgets.FloatSlider(
    value=0.01,
    min=0.01,
    max=5.0,
    step=0.01,
    description="Sigma PSF:",
    layout=widgets.Layout(width="800px"),
)

pixels_downsample_slider = widgets.IntSlider(
    value=64,
    min=10,
    max=64,
    step=1,
    description="Pixels downsampled to:",
    layout=widgets.Layout(width="800px"),
)

sigma_noise_slider = widgets.FloatSlider(
    value=0.01,
    min=0.01,
    max=0.2,
    step=0.0005,
    description="Noise σ:",
    layout=widgets.Layout(width="800px"),
)

image_selector = widgets.ToggleButtons(
    options=[("Image 1", 0), ("Image 2", 1), ("Image 3", 2), ("Image 4", 3), ("Image 5", 4)],
    description="Select image:",
    layout=widgets.Layout(width="800px"),
)

out_pipeline = widgets.Output()

# Global variables to reuse in later cells
galaxies_psf = None
galaxies_downsampled = None
galaxies_noisy = None

def update_pipeline(change=None):
    global galaxies_psf, galaxies_downsampled, galaxies_noisy

    with out_pipeline:
        clear_output(wait=True)

        sigma_psf_val = float(sigma_psf_slider.value)
        size_val = int(pixels_downsample_slider.value)
        sigma_n_val = float(sigma_noise_slider.value)

        galaxies_psf = psf_on_image(galaxies, sigma=sigma_psf_val)
        galaxies_downsampled = downsample_img(galaxies_psf, size=size_val)
        galaxies_noisy = add_gaussian_noise(galaxies_downsampled, sigma_n_val)

        selected_idx = image_selector.value
        show_grid_final(galaxies_noisy, selected_idx=selected_idx)

for w in [sigma_psf_slider, pixels_downsample_slider, sigma_noise_slider, image_selector]:
    w.observe(update_pipeline, names="value")

ui = widgets.VBox(
    [
        sigma_psf_slider,
        pixels_downsample_slider,
        sigma_noise_slider,
        image_selector,
        out_pipeline,
    ]
)

display(ui)
update_pipeline()


## 6. A 1-minute intro to diffusion models 🌫️➡️🌌

Our AI model is a **diffusion model**, a type of generative model that can *create* new galaxies.

The idea in words:

1. Start from a real galaxy image and **gradually add noise** until it looks like pure static.
2. Train a neural network to **reverse this process**, step by step removing noise and recovering structure.
3. After training, we can start from random noise and let the model **denoise its way** into a realistic galaxy.

In the next cell, we’ll sample directly from this prior to see what kinds of galaxies the model has learned to imagine 🎨.


In [ ]:
# Prior sampling code
prior_samples = model.sample(shape=(20, 1, 64, 64), device=DEVICE, steps=70)

fig, ax = plt.subplots(2, 10, figsize=(20, 4), dpi=200)
for i in range(2):
    for j in range(10):
        idx = i * 10 + j
        img = img_to_show(prior_samples[idx], log_scale=True)
        ax[i, j].imshow(img, cmap="magma")
        ax[i, j].axis("off")
plt.suptitle("Samples from the diffusion model prior", fontsize=16)
plt.tight_layout()
plt.show()

## 7. From blurry image back to sharp galaxy: the inverse problem 🔄

Now comes the main challenge:

> Given a **noisy, blurred, low-resolution image** of a galaxy, can we guess what the **underlying high-resolution galaxy** looked like?

To do this, we:

1. Build a **linear operator** `A` that applies the same telescope effects (PSF + downsampling) we used above.
2. Use our diffusion model as a **prior** (it knows what galaxies usually look like).
3. Run a sampling algorithm (`LinearGaussianPosteriorSampler`) that combines the data and the prior to generate possible high-resolution galaxies consistent with the observation.

This cell sets up `A`, chooses the galaxy you selected earlier, and runs the sampler to draw several posterior samples.

If we try to ‘undo’ the telescope with no prior knowledge about galaxies, what would go wrong?

In [ ]:
# Build the linear operator A corresponding to the current PSF + downsampling
sigma_psf_val = float(sigma_psf_slider.value)
S_val = int(pixels_downsample_slider.value)
y_lin, A = psf_downsample_build_A(galaxies, sigma_psf=sigma_psf_val, S=S_val)


# Pick the selected galaxy and its noisy observation
idx = image_selector.value
y_obs = galaxies_noisy[idx]    # (S,S)
sigma_n = float(sigma_noise_slider.value)

sampler = LinearGaussianPosteriorSampler(
    observation=y_obs,   # (S,S)
    A=A,
    model=model,
    sigma_n=sigma_n,
    C=1.0,
    M=0.0,
)

samples = sampler.run(
    n_samples=4,
    steps=100,
    progress=True,
    true=galaxies[idx],       # for optional trajectory plotting
    plot_trajectory=True,    # set True if you want the animated view
    trajectory_stride=5,
)

print("Samples shape:", samples.shape)  # (4,1,Hs,Hs)


## 8. Looking at the reconstruction 🎨

Time to inspect what the sampler produced:

- **Top row:** the true high-resolution galaxy (left) and a few **posterior samples** of what the galaxy might look like.
- **Middle row:** the actual **observed image** (left) and the **mock observations** obtained by passing each sample through our telescope model `A`.
- **Bottom row:** **residuals** = (observed − mock) / noise level.

If the method is working well, the mock observations should resemble the real observation, and the residuals should look like random noise (no obvious patterns).


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9), dpi=200)

# Row 1: true + posterior samples
true_img = img_to_show(galaxies[idx], log_scale=True)
axes[0, 0].imshow(true_img, cmap="magma")
axes[0, 0].set_title("True")
axes[0, 0].axis("off")

for i in range(4):
    img = img_to_show(samples[i], log_scale=True)
    axes[0, i + 1].imshow(img, cmap="magma")
    axes[0, i + 1].set_title(f"Sample {i+1}")
    axes[0, i + 1].axis("off")

# Row 2: observed + mock observations
obs_img = img_to_show(y_obs, log_scale=False)
axes[1, 0].imshow(obs_img, cmap="magma")
axes[1, 0].set_title("Observed")
axes[1, 0].axis("off")

for i in range(4):
    mock = samples[i].view(1, sampler.Msrc) @ A.t()  # (1,Mobs)
    mock_img = mock.view(1, 1, y_obs.shape[0], y_obs.shape[1])
    mock_img_to_show = img_to_show(mock_img[0, 0], log_scale=False)
    axes[1, i + 1].imshow(mock_img_to_show, cmap="magma")
    axes[1, i + 1].set_title(f"Mock Obs {i+1}")
    axes[1, i + 1].axis("off")

    # Row 3: residuals
    residual = (y_obs - mock_img[0, 0]) / sigma_n
    residual_img = img_to_show(residual, log_scale=False)
    axes[2, i + 1].imshow(residual_img, cmap="bwr", vmin=-3, vmax=3)
    axes[2, i + 1].set_title(f"Residual {i+1}")
    axes[2, i + 1].axis("off")

axes[2, 0].axis("off")
plt.tight_layout()
plt.show()


## 9. Mean image and uncertainty map 📊

Instead of looking at samples one by one, we can summarise the posterior with:

- The **posterior mean** (average of all samples) – a single “best guess” galaxy image.
- The **posterior standard deviation** – an **uncertainty map** showing where the model is more or less sure.

Compare:

- The true galaxy,
- The posterior mean,
- The uncertainty map.


In [ ]:
samples_mean = samples.mean(dim=0)  # (1,Hs,Hs)
samples_std = samples.std(dim=0)    # (1,Hs,Hs)

fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=200)

img_mean = img_to_show(samples_mean[0], log_scale=True)
axes[1].imshow(img_mean, cmap="magma")
axes[1].set_title("Posterior Mean")
axes[1].axis("off")

img_std = img_to_show(samples_std[0], log_scale=False)
axes[2].imshow(img_std, cmap="magma")
axes[2].set_title("Posterior Std Dev")
axes[2].axis("off")

true_img = img_to_show(galaxies[idx], log_scale=True)
axes[0].imshow(true_img, cmap="magma")
axes[0].set_title("True Image")
axes[0].axis("off")

plt.tight_layout()
plt.show()


## 10. The mystery galaxy challenge 🕵️‍♀️🌌

So far we’ve played with galaxies whose true images we secretly knew.  
Now let’s make things more exciting:

> Astronomers observed a **mystery galaxy** 12 times with a small telescope.  
> Each observation is noisy and blurry, and we do **not** look at the true high-resolution image until the very end.

Your mission:  
Use the same AI + physics pipeline to reconstruct what this galaxy most likely looks like. Ready? 🙂


### 10.1 The data: 12 noisy exposures 📷📷📷

In this cell we load the 12 telescope images of our mystery galaxy.

Each panel is:

- One independent **exposure** of the same object,
- With the **same telescope** and **same noise level**.

Right now you only see a noisy blob… but hidden inside there is a very distinctive shape. Look at the grid and try to imagine what kind of galaxy could hide behind all that noise.


In [ ]:
#! Do not change these values!!
sigma_n_ood = 0.1
sigma_psf_ood = 0.1
res_ood = 64

# Load observations of the mystery OOD galaxy
mistery_galaxy_obs = torch.load("super-resolution-workshop/data/mistery_galaxy_obs.pt")

fig, ax = plt.subplots(3, 4, figsize=(16, 12), dpi=200)
for i in range(12):
    img = img_to_show(mistery_galaxy_obs[i], log_scale=False)
    ax[i // 4, i % 4].imshow(img, cmap="magma")
    ax[i // 4, i % 4].set_title(f"Mistery galaxy - Observation {i+1}")
    ax[i // 4, i % 4].axis("off")
plt.tight_layout()
plt.show()


### 10.2 Building the telescope model for the mystery galaxy 🔧

To analyse this new target, we need the matching **forward model**:

- We build a new operator `A` that captures the PSF and resolution used for these observations.
- Applying `A` to a sharp galaxy gives us a simulated noisy-free version of what the telescope would see.

This is the same idea as before, but now the operator is calibrated specifically for the mystery galaxy data.


In [ ]:
sigma_n_ood = 0.1
sigma_psf_ood = 0.1
res_ood = 64

y_lin_ood, A = psf_downsample_build_A(
    mistery_galaxy_obs,
    sigma_psf=sigma_psf_ood,
    S=res_ood,
)

print("A shape:", A.shape)
print("y_lin shape:", y_lin_ood.shape)


### 10.3 Sampling possible true galaxies 🎲

Now we connect everything:

1. The **12 observations** (one per exposure) are combined in the likelihood.
2. The diffusion model provides a **prior** over realistic galaxies.
3. The sampler draws several **posterior samples** – different high-resolution galaxies that all fit the data.

Each sample is one plausible reconstruction of the mystery galaxy.  
Let’s run the sampler and see what it comes up with.


In [ ]:
sampler_mistery = LinearGaussianPosteriorSampler(
    observation=mistery_galaxy_obs,   # (B,res_ood,res_ood)
    A=A,
    model=model,
    sigma_n=sigma_n_ood,
    C=1.0,
    M=0.0,
)

samples_mistery = sampler_mistery.run(
    n_samples=4,
    steps=200,
    progress=True,
    true=None,
    plot_trajectory=True,
    trajectory_stride=5,
)

print("OOD samples shape:", samples_mistery.shape)  # (4,1,Hs,Hs)


### 10.4 Do our reconstructions explain the data? 👀

This figure mirrors what we did earlier:

- **Top row:** four posterior samples of the *unknown* true galaxy (we still hide the real one!).
- **Middle row:** one of the observed images and the **mock observations** generated from each sample.
- **Bottom row:** residuals for that exposure.

Check whether:

- The mock observations look similar to the real one,
- The residuals mostly look like random noise.

If yes, our AI + physics model is giving a very good explanation of the measurements.


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9), dpi=200)

# Top row: OOD posterior samples
axes[0, 0].axis("off")
for i in range(4):
    img = img_to_show(samples_mistery[i], log_scale=True, min_val=0.1)
    axes[0, i + 1].imshow(img, cmap="magma")
    axes[0, i + 1].set_title(f"Sample {i+1}")
    axes[0, i + 1].axis("off")

# Middle row: Mistery galaxy observations and mock observations
obs_img = img_to_show(mistery_galaxy_obs[0], log_scale=False)
axes[1, 0].imshow(obs_img, cmap="magma")
axes[1, 0].set_title("Observed")
axes[1, 0].axis("off")

for i in range(4):
    mock = samples_mistery[i].view(1, sampler_mistery.Msrc) @ A.t()  # (1,Mobs)
    mock_img = mock.view(1, 1, mistery_galaxy_obs.shape[1], mistery_galaxy_obs.shape[2])
    mock_img_to_show = img_to_show(mock_img[0, 0], log_scale=False)
    axes[1, i + 1].imshow(mock_img_to_show, cmap="magma")
    axes[1, i + 1].set_title(f"Mock Obs {i+1}")
    axes[1, i + 1].axis("off")

    residual = (mistery_galaxy_obs[0] - mock_img[0, 0]) / sigma_n_ood
    residual_img = img_to_show(residual, log_scale=False)
    axes[2, i + 1].imshow(residual_img, cmap="bwr", vmin=-3, vmax=3)
    axes[2, i + 1].set_title(f"Residual {i+1}")
    axes[2, i + 1].axis("off")

# Question mark for unknown true image
axes[0, 0].text(0.5, 0.5, "?", fontsize=40, ha="center", va="center")
axes[0, 0].set_title("True (unknown)")

axes[2, 0].axis("off")
plt.tight_layout()
plt.show()


### 10.5 The big reveal 🎉🍁

Time to lift the curtain!

In this cell we finally load the **true high-resolution image** of the mystery galaxy and compare it to our posterior samples.

Surprise: the “galaxy” was designed to look like a **maple leaf** – a little Montréal Easter egg 🍁

Look carefully:

- Do the bright clumps and the overall **leaf shape** in the samples match the true image?
- Are there parts of the leaf (stem, tips, sides) that some samples capture better than others?

This shows how our AI + physics model can recover a very specific shape from noisy, blurred data.


In [ ]:
true_mistery = torch.load(
    "super-resolution-workshop/data/true_mistery_galaxy.pt"
).to(DEVICE)

# Plot posterior samples against the true mistery galaxy
fig, axes = plt.subplots(1, 5, figsize=(15, 3), dpi=200)
true_img = img_to_show(true_mistery[0], log_scale=True, min_val=0.1)
axes[0].imshow(true_img, cmap="magma")
axes[0].set_title("True Mistery Galaxy")
axes[0].axis("off")

for i in range(4):
    img = img_to_show(samples_mistery[i], log_scale=True, min_val=0.1)
    axes[i + 1].imshow(img, cmap="magma")
    axes[i + 1].set_title(f"Sample {i+1}")
    axes[i + 1].axis("off")
plt.tight_layout()
plt.show()

### 10.6 Mean reconstruction and final comparison ✅

Here we compare three images side by side:

1. The **true maple-leaf galaxy**,
2. The **posterior mean** (average over all samples),
3. One of the **noisy observed exposures**.

Notice how:

- The observation is so noisy that the maple-leaf shape is almost invisible,
- The posterior mean is much sharper and clearly shows the **leaf outline and stem**,
- Some tiny details still differ, reminding us that we always have **uncertainty**.

From a grainy blob to a recognizable maple leaf using data + a learned prior. You’ve just used a state-of-the-art generative model to improve telescope images and solve a real inverse problem! 🛰️🍁

In [ ]:
samples_mistery_mean = samples_mistery.mean(dim=0)  # (1,Hs,Hs)

fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=200)

true_img = img_to_show(true_mistery, log_scale=True, min_val=0.2)
axes[0].imshow(true_img, cmap="magma")
axes[0].set_title("True Image")
axes[0].axis("off")

img_mean = img_to_show(samples_mistery_mean, log_scale=True, min_val=0.2)
axes[1].imshow(img_mean, cmap="magma")
axes[1].set_title("Posterior Mean")
axes[1].axis("off")

# Stacking result
axes[2].imshow(img_to_show(mistery_galaxy_obs[0], log_scale=False), cmap="magma")
axes[2].set_title("Observed (1st exposure)")
axes[2].axis("off")
plt.tight_layout()
plt.show()
